# Auditory Scene Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensein/senselab/blob/alpha/tutorials/audio/auditory_scene_analysis.ipynb)

Classify audio events over time using a sliding-window approach. This tutorial
shows how to detect *what* is happening in a recording (speech, music, dog bark,
siren, etc.) and *when* it happens, using the Audio Spectrogram Transformer
fine-tuned on AudioSet (521 event classes).

In [ ]:
# Install senselab and dependencies
!pip install -q uv
!uv pip install --pre --system "senselab"

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages (e.g., numpy) that are already loaded.
> Go to **Runtime → Restart session**, then **run all cells below** (skip this install cell).

In [ ]:
import os
from pathlib import Path

import torch

from senselab.audio.data_structures import Audio
from senselab.audio.tasks.classification.api import classify_audios_in_windows, scene_results_to_segments
from senselab.audio.tasks.plotting.plotting import play_audio, plot_aligned_panels
from senselab.audio.tasks.preprocessing.preprocessing import downmix_audios_to_mono, resample_audios
from senselab.utils.data_structures import DeviceType, HFModel

%matplotlib inline

device = DeviceType.CUDA if torch.cuda.is_available() else DeviceType.CPU
print(f"Using device: {device.value}")

## 1. Load Audio

In [ ]:
# Record audio in Google Colab (uses browser microphone)
# If not in Colab, skip this cell and run the next one to load a sample file.
Path("tutorial_audio_files").mkdir(exist_ok=True)

try:
    from google.colab import output
    from IPython.display import HTML, display

    RECORD_JS = """
    <button id="record-btn" style="font-size:16px;padding:8px 16px">🎙 Click to Record (5s)</button>
    <script>
    (async () => {
        const btn = document.getElementById('record-btn');
        btn.onclick = async () => {
            btn.textContent = '🔴 Recording...';
            const stream = await navigator.mediaDevices.getUserMedia({audio: true});
            const recorder = new MediaRecorder(stream);
            const chunks = [];
            recorder.ondataavailable = e => chunks.push(e.data);
            recorder.onstop = async () => {
                stream.getTracks().forEach(t => t.stop());
                const blob = new Blob(chunks, {type: 'audio/wav'});
                const reader = new FileReader();
                reader.onload = () => {
                    const b64 = reader.result.split(',')[1];
                    google.colab.kernel.invokeFunction('save_recording', [b64], {});
                };
                reader.readAsDataURL(blob);
                btn.textContent = '✅ Recorded! Run the next cell to load it.';
            };
            recorder.start();
            setTimeout(() => recorder.stop(), 5000);
        };
    })();
    </script>
    """

    def save_recording(b64_data):
        import base64

        raw = base64.b64decode(b64_data)
        with open("tutorial_audio_files/my_recording.wav", "wb") as f:
            f.write(raw)

    output.register_callback("save_recording", save_recording)
    display(HTML(RECORD_JS))
    print("Click the button above to record 5 seconds of audio.")
    print("After recording, run the NEXT cell to load it.")
except Exception:
    print("Recording not available (not in Colab). Run the next cell to load a sample file.")

In [ ]:
import urllib.request

base_url = "https://github.com/sensein/senselab/raw/main/src/tests/data_for_testing"

# Always download sample file
Path("tutorial_audio_files").mkdir(exist_ok=True)
sample_dest = Path("tutorial_audio_files/audio_48khz_mono_16bits.wav")
if not sample_dest.exists():
    urllib.request.urlretrieve(f"{base_url}/audio_48khz_mono_16bits.wav", str(sample_dest))

# Check if a recording was made
rec_path = Path("tutorial_audio_files/my_recording.wav")
if rec_path.exists() and rec_path.stat().st_size > 1000:
    audio_path = str(rec_path.resolve())
    print(f"Using your recording: {audio_path}")
else:
    audio_path = str(sample_dest.resolve())
    print(f"Using sample audio: {audio_path}")

audio = Audio(filepath=audio_path)
print(f"Loaded: {audio.waveform.shape[1]} samples at {audio.sampling_rate} Hz")
print(f"Duration: {audio.waveform.shape[1] / audio.sampling_rate:.2f} seconds")

# Ensure mono, resample to 16 kHz
audio_mono = downmix_audios_to_mono([audio])[0]
audio_16k = resample_audios([audio_mono], resample_rate=16000)[0]
print(f"Resampled to {audio_16k.sampling_rate} Hz, mono")

In [ ]:
play_audio(audio_16k)

## 2. Windowed Scene Classification

Instead of classifying the entire recording as one label, we slide a **1-second
window** across the audio with a **0.5-second hop** and classify each window
independently. This reveals *when* different acoustic events occur.

We use the **Audio Spectrogram Transformer (AST)** fine-tuned on AudioSet, which
can recognize 521 event classes (speech, music, environmental sounds, etc.).

In [ ]:
model = HFModel(path_or_uri="MIT/ast-finetuned-audioset-10-10-0.4593")
results = classify_audios_in_windows(
    [audio_16k], model=model, window_size=1.0, hop_size=0.5, top_k=5, device=device
)

# Show first few windows
print(f"Total windows: {len(results[0])}\n")
for r in results[0][:5]:
    print(f"{r['start']:.1f}-{r['end']:.1f}s: {r['labels'][0]} ({r['scores'][0]:.3f})")

## 3. Timeline Visualization

Convert the per-window top-1 labels into a segment timeline and plot alongside
the waveform and spectrogram using `plot_aligned_panels`.

In [ ]:
segments = scene_results_to_segments(results[0])
plot_aligned_panels(
    audio_16k,
    [
        {"type": "waveform"},
        {"type": "spectrogram", "mel": False},
        {"type": "segments", "segments": segments},
    ],
    title="Auditory Scene Analysis",
)

## 4. Filtering by Event Type

You can filter the results to find windows where a specific event appears in the
top predictions. For example, find all windows containing speech:

In [ ]:
speech_windows = [r for r in results[0] if "Speech" in r["labels"][:3]]
print(f"Found {len(speech_windows)} windows with speech in top-3 predictions")
for w in speech_windows[:5]:
    print(f"  {w['start']:.1f}-{w['end']:.1f}s")

## Summary

| Step | Function | Output |
|------|----------|--------|
| Windowed classification | `classify_audios_in_windows` | Per-window labels + scores |
| Segment conversion | `scene_results_to_segments` | Timeline segments for plotting |
| Visualization | `plot_aligned_panels` | Waveform + spectrogram + event timeline |
| Filtering | List comprehension on results | Windows matching specific events |

**Key parameters:**
- `window_size` (default 1.0s) — longer windows give more context but less temporal precision
- `hop_size` (default 0.5s) — smaller hops give finer time resolution but more computation
- `top_k` (default 5) — how many labels to keep per window

**Model:** `MIT/ast-finetuned-audioset-10-10-0.4593` covers 521 AudioSet classes.
For domain-specific tasks (e.g., bird species, urban sounds), swap in a
specialized model from the [HuggingFace Hub](https://huggingface.co/models?pipeline_tag=audio-classification).